## Setup & Imports

In [ ]:
# Uncomment when running for the first time
# !pip install transformers datasets accelerate scikit-learn seaborn
# !pip install scispacy
# !pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_ner_bc5cdr_md-0.5.3.tar.gz
# The en_ner_bc5cdr_md model is trained on BC5CDR — covers diseases & chemicals
print("Dependencies ready.")


In [ ]:
import os, re, json, warnings, pickle
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from transformers import (
 AutoTokenizer, AutoModelForSequenceClassification,
 AutoModelForTokenClassification,
 TrainingArguments, Trainer,
 EarlyStoppingCallback, pipeline
)
from datasets import Dataset, DatasetDict

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
 accuracy_score, f1_score,
 classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Device: {DEVICE}")
if DEVICE.type == 'cuda':
 print(f" GPU: {torch.cuda.get_device_name(0)}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_CSV = "drugsComTrain_raw.csv"
TEST_CSV = "drugsComTest_raw.csv"
MODEL_DIR = Path("./saved_models")
MODEL_DIR.mkdir(exist_ok=True)

SENTIMENT_MODEL_CKPT = "roberta-base" # RoBERTa for sentiment
NER_MODEL_CKPT = "allenai/scibert_scivocab_uncased" # fallback NER
# Primary NER via SciSpaCy: en_ner_bc5cdr_md (install above)

print(" Setup complete.")


## Data Loading & Cleaning

In [ ]:
df_train_raw = pd.read_csv(TRAIN_CSV)
df_test_raw = pd.read_csv(TEST_CSV)

print(f"Train: {df_train_raw.shape} | Test: {df_test_raw.shape}")
df_train_raw.head(3)


In [ ]:
HTML_RE = re.compile(r'<[^>]+>') # HTML tags
AMP_RE = re.compile(r'&amp;|&#?\w+;') # HTML entities
URL_RE = re.compile(r'https?://\S+')
MULTI_SP = re.compile(r'\s{2,}')

def clean_review(text: str) -> str:
 """Light cleaning — preserve medical vocabulary for transformer tokenizer."""
 if not isinstance(text, str):
 return ""
 text = HTML_RE.sub(' ', text) # remove HTML tags
 text = AMP_RE.sub(' ', text) # decode HTML entities
 text = URL_RE.sub(' ', text) # drop URLs
 text = text.replace('\\r', ' ').replace('\\n', ' ') # escaped newlines
 text = MULTI_SP.sub(' ', text).strip()
 # NOTE: We do NOT lowercase or strip punctuation — RoBERTa uses casing
 # and punctuation for contextual embeddings.
 return text

def rating_to_sentiment(rating: int) -> int:
 """Map 1-10 rating → 3-class label.
 0=Negative (1-4), 1=Neutral (5-6), 2=Positive (7-10)
 """
 if rating <= 4: return 0 # Negative
 elif rating <= 6: return 1 # Neutral
 else: return 2 # Positive

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
 df = df.copy()
 df['review_clean'] = df['review'].apply(clean_review)
 df['sentiment'] = df['rating'].apply(rating_to_sentiment)
 df['drugName'] = df['drugName'].str.strip().str.lower()
 df['condition'] = df['condition'].fillna('unknown').str.lower()
 # Drop rows with empty reviews
 df = df[df['review_clean'].str.len() > 20].reset_index(drop=True)
 return df

df_train = preprocess(df_train_raw)
df_test = preprocess(df_test_raw)

LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
LABEL_NAMES = [LABEL_MAP[i] for i in range(3)]

print(f"Clean train: {df_train.shape} | Clean test: {df_test.shape}")
print("\nSentiment distribution (train):")
print(df_train['sentiment'].map(LABEL_MAP).value_counts())


## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Rating histogram
axes[0].hist(df_train['rating'], bins=10, color='steelblue', edgecolor='white')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating (1–10)')

# Sentiment class balance
counts = df_train['sentiment'].map(LABEL_MAP).value_counts()
colors = ['#e74c3c', '#95a5a6', '#2ecc71']
axes[1].bar(counts.index, counts.values, color=colors)
axes[1].set_title('Sentiment Class Balance')
for i, v in enumerate(counts.values):
 axes[1].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_class_balance.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Class imbalance ratio (max/min): {counts.max()/counts.min():.1f}x")


In [ ]:
df_train['review_len'] = df_train['review_clean'].str.split().str.len()

plt.figure(figsize=(9, 3))
plt.hist(df_train['review_len'].clip(upper=500), bins=60, color='mediumpurple', edgecolor='white')
plt.axvline(512, color='red', linestyle='--', label='RoBERTa max (512 tok)')
plt.title('Review Word-Count Distribution')
plt.xlabel('Word count')
plt.legend()
plt.tight_layout()
plt.show()

p95 = df_train['review_len'].quantile(0.95)
print(f"95th-percentile word count: {p95:.0f} → MAX_LEN=256 is safe for most reviews.")


In [ ]:
top_drugs = df_train['drugName'].value_counts().head(15)

plt.figure(figsize=(11, 4))
sns.barplot(x=top_drugs.values, y=top_drugs.index, orient='h', palette='Blues_r')
plt.title('Top 15 Drugs by Review Count (Train)')
plt.xlabel('Number of reviews')
plt.tight_layout()
plt.show()


## Sentiment Modeling — RoBERTa Fine-Tune

In [ ]:
# Comment out SAMPLE_N line to train on full dataset
SAMPLE_N = 20_000
MAX_LEN = 256

if SAMPLE_N:
 df_sub = df_train.groupby('sentiment', group_keys=False).apply(
 lambda g: g.sample(min(len(g), SAMPLE_N // 3), random_state=SEED)
 ).reset_index(drop=True)
else:
 df_sub = df_train.copy()

# Train / validation split (stratified)
X = df_sub['review_clean'].tolist()
y = df_sub['sentiment'].tolist()
X_tr, X_val, y_tr, y_val = train_test_split(
 X, y, test_size=0.15, stratify=y, random_state=SEED
)

print(f"Train samples: {len(X_tr)} | Val samples: {len(X_val)}")
print(f"Val class distribution: {Counter(y_val)}")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_CKPT)

def tokenize_fn(examples):
 return tokenizer(
 examples['text'],
 truncation=True,
 padding='max_length',
 max_length=MAX_LEN
 )

def build_hf_dataset(texts, labels):
 ds = Dataset.from_dict({'text': texts, 'label': labels})
 return ds.map(tokenize_fn, batched=True, remove_columns=['text'])

ds_train = build_hf_dataset(X_tr, y_tr)
ds_val = build_hf_dataset(X_val, y_val)

# Test-set tokenisation (for final eval)
ds_test = build_hf_dataset(
 df_test['review_clean'].tolist(),
 df_test['sentiment'].tolist()
)

print("Datasets ready:", ds_train, ds_val)


In [ ]:
classes = np.unique(y_tr)
cw = compute_class_weight('balanced', classes=classes, y=y_tr)
class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)
print("Class weights:", dict(zip(LABEL_NAMES, cw.round(3))))


In [ ]:
from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
 """Overrides compute_loss to apply class weights."""
 def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
 labels = inputs.pop('labels')
 outputs = model(**inputs)
 logits = outputs.logits
 loss_fn = CrossEntropyLoss(weight=class_weights)
 loss = loss_fn(logits, labels)
 return (loss, outputs) if return_outputs else loss


In [ ]:
def compute_metrics(eval_pred):
 logits, labels = eval_pred
 preds = np.argmax(logits, axis=-1)
 return {
 'accuracy': accuracy_score(labels, preds),
 'f1_macro': f1_score(labels, preds, average='macro'),
 'f1_weighted': f1_score(labels, preds, average='weighted'),
 }


In [ ]:
SENTIMENT_OUT = MODEL_DIR / 'sentiment_roberta'

training_args = TrainingArguments(
 output_dir=str(SENTIMENT_OUT),
 num_train_epochs=5,
 per_device_train_batch_size=16,
 per_device_eval_batch_size=32,
 learning_rate=2e-5, # sweet-spot for RoBERTa fine-tune
 weight_decay=0.01, # L2 regularization
 warmup_ratio=0.1, # 10% warmup steps
 lr_scheduler_type='cosine', # cosine decay
 eval_strategy='epoch',
 save_strategy='epoch',
 load_best_model_at_end=True,
 metric_for_best_model='f1_macro',
 greater_is_better=True,
 logging_steps=100,
 fp16=DEVICE.type == 'cuda', # mixed precision on GPU
 dataloader_num_workers=2,
 report_to='none', # disable wandb / tensorboard
 seed=SEED,
)

id2label = {i: l for i, l in LABEL_MAP.items()}
label2id = {l: i for i, l in id2label.items()}

model_sentiment = AutoModelForSequenceClassification.from_pretrained(
 SENTIMENT_MODEL_CKPT,
 num_labels=3,
 id2label=id2label,
 label2id=label2id,
).to(DEVICE)

trainer = WeightedTrainer(
 model=model_sentiment,
 args=training_args,
 train_dataset=ds_train,
 eval_dataset=ds_val,
 compute_metrics=compute_metrics,
 callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training...")
trainer.train()


In [ ]:
val_results = trainer.evaluate(ds_val)
print("\n Validation Results:")
for k, v in val_results.items():
 if 'eval_' in k:
 print(f" {k.replace('eval_','')}: {v:.4f}")


## Side Effect Extraction — Hybrid NER Pipeline

In [ ]:
import spacy

try:
 nlp_ner = spacy.load('en_ner_bc5cdr_md')
 print(" SciSpaCy BC5CDR model loaded.")
 NER_AVAILABLE = True
except OSError:
 print(" SciSpaCy model not found. Run:")
 print(" pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_ner_bc5cdr_md-0.5.3.tar.gz")
 NER_AVAILABLE = False
 # Fallback: keyword-only mode
 nlp_ner = None


In [ ]:
# A broad vocabulary of side effects — augments NER on informal text
SIDE_EFFECT_KEYWORDS = {
 # Gastrointestinal
 'nausea', 'vomiting', 'diarrhea', 'constipation', 'stomach pain',
 'bloating', 'indigestion', 'heartburn', 'abdominal pain', 'cramps',
 # Neurological
 'headache', 'migraine', 'dizziness', 'drowsiness', 'insomnia',
 'fatigue', 'brain fog', 'confusion', 'anxiety', 'depression',
 'mood swings', 'irritability', 'tremors', 'seizures',
 # Cardiovascular
 'palpitations', 'heart racing', 'chest pain', 'shortness of breath',
 'high blood pressure', 'low blood pressure', 'swelling',
 # Musculoskeletal
 'muscle pain', 'joint pain', 'muscle weakness', 'cramps', 'aches',
 # Dermatological
 'rash', 'itching', 'hives', 'dry skin', 'hair loss', 'acne',
 # Metabolic
 'weight gain', 'weight loss', 'appetite loss', 'increased appetite',
 'dry mouth', 'thirst', 'sweating',
 # Urinary / Reproductive
 'frequent urination', 'urinary retention', 'sexual dysfunction',
 'decreased libido',
 # General
 'fever', 'chills', 'infection', 'weakness', 'lethargy',
 'blurred vision', 'tinnitus', 'back pain'
}

def extract_keyword_effects(text: str) -> list:
 """Match side-effect keywords (multi-word phrases first)."""
 text_lower = text.lower()
 found = []
 for kw in sorted(SIDE_EFFECT_KEYWORDS, key=len, reverse=True):
 if kw in text_lower:
 found.append(kw)
 return found


In [ ]:
def extract_side_effects(text: str) -> list:
 """
 Hybrid approach:
 1. SciSpaCy NER → DISEASE entities
 2. Keyword match → curated vocabulary
 Returns deduplicated list of side effects.
 """
 effects = []

 # Stage 1: NER (if available)
 if NER_AVAILABLE and nlp_ner:
 doc = nlp_ner(text[:1000]) # cap for speed
 for ent in doc.ents:
 if ent.label_ == 'DISEASE':
 effects.append(ent.text.lower().strip())

 # Stage 2: Keyword fallback (always run as supplement)
 kw_effects = extract_keyword_effects(text)
 effects.extend(kw_effects)

 seen, unique = set(), []
 for e in effects:
 if e not in seen and len(e) > 2:
 seen.add(e)
 unique.append(e)
 return unique


def build_drug_side_effect_profile(df: pd.DataFrame) -> dict:
 """
 For each drug, aggregate extracted side effects across all reviews.
 Returns {drug_name: Counter({effect: freq})}
 """
 drug_effects = defaultdict(Counter)

 for _, row in df.iterrows():
 drug = row['drugName']
 review = row['review_clean']
 effects = extract_side_effects(review)
 drug_effects[drug].update(effects)

 return dict(drug_effects)


print(" Building side effect profile (this may take a few minutes)...")
# Use full train + test for richer profiles
df_all = pd.concat([df_train, df_test], ignore_index=True)

TOP_DRUGS_N = 500 # set None to process all drugs
if TOP_DRUGS_N:
 top_drug_names = df_all['drugName'].value_counts().head(TOP_DRUGS_N).index
 df_all_sub = df_all[df_all['drugName'].isin(top_drug_names)]
else:
 df_all_sub = df_all

DRUG_EFFECT_PROFILE = build_drug_side_effect_profile(df_all_sub)
print(f" Profiles built for {len(DRUG_EFFECT_PROFILE)} drugs.")


In [ ]:
example_drug = 'lexapro'
if example_drug in DRUG_EFFECT_PROFILE:
 top5 = DRUG_EFFECT_PROFILE[example_drug].most_common(5)
 print(f"Top 5 side effects for '{example_drug}':")
 for rank, (effect, count) in enumerate(top5, 1):
 print(f" {rank}. {effect.capitalize():<25} (mentioned {count}x)")
else:
 print(f"'{example_drug}' not found in profile. Available sample:")
 print(list(DRUG_EFFECT_PROFILE.keys())[:10])


## Evaluation

In [ ]:
print("Running inference on test set...")

test_preds_output = trainer.predict(ds_test)
y_pred = np.argmax(test_preds_output.predictions, axis=-1)
y_true = df_test['sentiment'].values

print("\n Classification Report:")
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, data, fmt, title in zip(
 axes,
 [cm, cm_pct],
 ['d', '.2%'],
 ['Confusion Matrix (counts)', 'Confusion Matrix (%)'],
):
 sns.heatmap(
 data, annot=True, fmt=fmt, cmap='Blues',
 xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
 ax=ax, linewidths=0.5
 )
 ax.set_xlabel('Predicted')
 ax.set_ylabel('True')
 ax.set_title(title)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
from sklearn.metrics import f1_score as f1
per_class_f1 = f1(y_true, y_pred, average=None)

plt.figure(figsize=(7, 3))
bars = plt.bar(LABEL_NAMES, per_class_f1, color=['#e74c3c','#95a5a6','#2ecc71'])
for bar, val in zip(bars, per_class_f1):
 plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
 f'{val:.3f}', ha='center', fontweight='bold')
plt.ylim(0, 1.1)
plt.title('F1-Score by Sentiment Class')
plt.ylabel('F1-Score')
plt.tight_layout()
plt.show()


In [ ]:
sample_idx = [0, 5, 10, 20, 50]
print("Sample predictions on test set:\n")
print(f"{'Review (truncated)':<60} {'True':<12} {'Predicted':<12}")
print("-" * 90)
for i in sample_idx:
 review_short = df_test.iloc[i]['review_clean'][:57] + '...'
 true_lbl = LABEL_MAP[y_true[i]]
 pred_lbl = LABEL_MAP[y_pred[i]]
 match = '' if true_lbl == pred_lbl else ''
 print(f"{review_short:<60} {true_lbl:<12} {pred_lbl:<12} {match}")


## Unified Inference Pipeline

In [ ]:
sentiment_pipeline = pipeline(
 task='text-classification',
 model=model_sentiment,
 tokenizer=tokenizer,
 device=0 if DEVICE.type == 'cuda' else -1,
 truncation=True,
 max_length=MAX_LEN,
)

print("Sentiment pipeline ready.")


In [ ]:
def aggregate_drug_sentiment(drug_name: str, df: pd.DataFrame,
 batch_size: int = 32) -> dict:
 """
 Run sentiment classifier on all reviews for a drug.
 Returns majority vote + confidence scores.
 """
 drug_lower = drug_name.strip().lower()
 reviews = df[df['drugName'] == drug_lower]['review_clean'].tolist()

 if not reviews:
 return {'error': f"No reviews found for '{drug_name}'"}

 # Batch inference
 results = sentiment_pipeline(reviews, batch_size=batch_size)

 label_counts = Counter(r['label'] for r in results)
 total = len(results)

 dominant = label_counts.most_common(1)[0][0]
 confidence_dist = {
 lbl: round(cnt / total, 3) for lbl, cnt in label_counts.items()
 }

 return {
 'drug': drug_name,
 'n_reviews': total,
 'sentiment': dominant,
 'confidence_distribution': confidence_dist,
 }


def predict(drug_name: str, df: pd.DataFrame = df_all,
 top_n: int = 5) -> dict:
 """
 Unified prediction pipeline.

 Parameters
 ----------
 drug_name : str — e.g. 'Lexapro', 'metformin'
 df : pd.DataFrame — corpus (default: full train+test)
 top_n : int — number of side effects to return

 Returns
 -------
 dict with keys:
 drug, n_reviews, sentiment, confidence_distribution,
 top_side_effects → list of (effect, count) tuples
 """
 drug_lower = drug_name.strip().lower()

 sentiment_info = aggregate_drug_sentiment(drug_lower, df)
 if 'error' in sentiment_info:
 return sentiment_info

 if drug_lower in DRUG_EFFECT_PROFILE:
 top_effects = DRUG_EFFECT_PROFILE[drug_lower].most_common(top_n)
 else:
 # Build on-the-fly for unknown drugs
 drug_reviews = df[df['drugName'] == drug_lower]['review_clean'].tolist()
 counter = Counter()
 for rev in drug_reviews:
 counter.update(extract_side_effects(rev))
 top_effects = counter.most_common(top_n)

 return {
 **sentiment_info,
 'top_side_effects': [
 {'rank': i+1, 'effect': eff, 'mentions': cnt}
 for i, (eff, cnt) in enumerate(top_effects)
 ]
 }


print(" predict() function ready.")


In [ ]:
def pretty_print(result: dict):
 if 'error' in result:
 print(f" {result['error']}")
 return
 sent_icon = {'Positive': '', 'Neutral': '', 'Negative': ''}.get(result['sentiment'], '')
 print(f"\n{'='*55}")
 print(f" Drug : {result['drug'].upper()}")
 print(f" Reviews : {result['n_reviews']}")
 print(f" Sentiment : {sent_icon} {result['sentiment']}")
 print(f" Distribution: {result['confidence_distribution']}")
 print(f" Top Side Effects:")
 for se in result.get('top_side_effects', []):
 print(f" {se['rank']}. {se['effect'].capitalize():<30} ({se['mentions']} mentions)")
 print('='*55)

# Run on a few drugs
for drug in ['lexapro', 'metformin', 'adderall']:
 pretty_print(predict(drug))


## Saving Models

In [ ]:
SENTIMENT_OUT.mkdir(parents=True, exist_ok=True)
model_sentiment.save_pretrained(str(SENTIMENT_OUT))
tokenizer.save_pretrained(str(SENTIMENT_OUT))
print(f" Sentiment model saved → {SENTIMENT_OUT}")

PROFILE_PATH = MODEL_DIR / 'drug_effect_profiles.pkl'
with open(PROFILE_PATH, 'wb') as f:
 pickle.dump(DRUG_EFFECT_PROFILE, f)
print(f" Side-effect profiles saved → {PROFILE_PATH}")

META_PATH = MODEL_DIR / 'metadata.json'
with open(META_PATH, 'w') as f:
 json.dump({
 'id2label': {str(k): v for k, v in id2label.items()},
 'label2id': label2id,
 'max_len': MAX_LEN,
 'base_model': SENTIMENT_MODEL_CKPT
 }, f, indent=2)
print(f" Metadata saved → {META_PATH}")


In [ ]:
# Demonstrate reloading from disk (production use-case)
loaded_tokenizer = AutoTokenizer.from_pretrained(str(SENTIMENT_OUT))
loaded_model = AutoModelForSequenceClassification.from_pretrained(str(SENTIMENT_OUT))

with open(PROFILE_PATH, 'rb') as f:
 loaded_profiles = pickle.load(f)

print(" Models reloaded successfully.")
print(f" Vocabulary size : {loaded_tokenizer.vocab_size}")
print(f" Labels : {loaded_model.config.id2label}")
print(f" Drug profiles : {len(loaded_profiles)} drugs")


In [ ]:
test_acc = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred, average='macro')

print("\n" + "="*55)
print(" FINAL RESULTS SUMMARY")
print("="*55)
print(f" Base model : {SENTIMENT_MODEL_CKPT}")
print(f" Test Accuracy : {test_acc:.4f}")
print(f" Test Macro-F1 : {test_f1:.4f}")
print(f" NER mode : {'SciSpaCy BC5CDR' if NER_AVAILABLE else 'Keyword-only fallback'}")
print(f" Drugs with profiles : {len(DRUG_EFFECT_PROFILE)}")
print("="*55)
print("\nUsage:")
print(' result = predict("lexapro")')
print(' # Returns: sentiment + top-5 side effects')
